In [1]:

!apt-get update
!apt-get install -y iverilog

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,772 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,302 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,643 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelea

In [53]:
SPEC = """
Design a synthesizable Verilog/SystemVerilog module named binary_to_bcd_converter with:
- input  [7:0]  binary_input
- output [11:0] bcd_output

bcd_output format: {hundreds[3:0], tens[3:0], ones[3:0]}.
For binary_input 0..255, convert to decimal and output BCD digits.
Purely combinational (no clocks).
Use a synthesizable double-dabble (shift-add-3) implementation.
"""

PROMPT = f"""
You are an expert digital design engineer.

Task: Write synthesizable Verilog code that compiles with:
iverilog -g2012

CRITICAL OUTPUT RULES (you MUST follow these exactly):
1) Your ENTIRE response must be ONLY a single fenced code block: ```verilog ... ```
2) NO explanations, NO prose, NO comments outside the code block.
3) NO manual step-by-step solving. Output CODE only.
4) The code block must contain ONLY the module binary_to_bcd_converter — no testbench.
5) Purely combinational. No latches.
6) Use always @* with a compact for-loop (not unrolled stages).
7) Use a single shift_reg [11:0] and integer loop variable i.

If you find yourself writing any text outside the code block, STOP and restart with just the code block.

Spec:
{SPEC}
""".strip()

In [54]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini model
model = genai.GenerativeModel('gemini-2.5-flash')
completion = model.generate_content(verilog_generation_prompt)
print(completion.text)

```verilog
module binary_to_bcd_converter (
    input  [7:0]  binary_input,
    output [11:0] bcd_output
);

    reg [11:0] shift_reg;
    integer i;

    always @* begin
        shift_reg = 12'b0; // Initialize BCD digits to zero

        // Iterate through each bit of the binary_input, from MSB to LSB
        for (i = 0; i < 8; i = i + 1) begin
            // Apply +3 correction to any BCD digit that is 5 or greater
            // This is done before the shift operation for the current stage

            // Hundreds digit (bits 11:8)
            if (shift_reg[11:8] >= 4'd5) begin
                shift_reg[11:8] = shift_reg[11:8] + 4'd3;
            end
            // Tens digit (bits 7:4)
            if (shift_reg[7:4] >= 4'd5) begin
                shift_reg[7:4] = shift_reg[7:4] + 4'd3;
            end
            // Ones digit (bits 3:0)
            if (shift_reg[3:0] >= 4'd5) begin
                shift_reg[3:0] = shift_reg[3:0] + 4'd3;
            end

            // Shift the e

In [56]:
import os, re, json, textwrap, subprocess
import requests

In [57]:
def extract_sv_code(text: str) -> str:
    # Prefer fenced systemverilog blocks
    m = re.search(r"```systemverilog\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if not m:
        # Fallback: any fenced block
        m = re.search(r"```(?:\w+)?\s*(.*?)```", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No fenced code block found in Claude output.")

    code = m.group(1).strip()

    # Sanity checks aligned with testbench expectations
    if "module binary_to_bcd_converter" not in code:
        raise ValueError("Extracted code does not contain 'module binary_to_bcd_converter'.")
    if "endmodule" not in code:
        raise ValueError("Extracted code does not contain 'endmodule'.")

    return code


sv_code = extract_sv_code(completion.text)
print("EXTRACTED SYSTEMVERILOG (preview)")
print(sv_code)

EXTRACTED SYSTEMVERILOG (preview)
module binary_to_bcd_converter (
    input  [7:0]  binary_input,
    output [11:0] bcd_output
);

    reg [11:0] shift_reg;
    integer i;

    always @* begin
        shift_reg = 12'b0; // Initialize BCD digits to zero

        // Iterate through each bit of the binary_input, from MSB to LSB
        for (i = 0; i < 8; i = i + 1) begin
            // Apply +3 correction to any BCD digit that is 5 or greater
            // This is done before the shift operation for the current stage

            // Hundreds digit (bits 11:8)
            if (shift_reg[11:8] >= 4'd5) begin
                shift_reg[11:8] = shift_reg[11:8] + 4'd3;
            end
            // Tens digit (bits 7:4)
            if (shift_reg[7:4] >= 4'd5) begin
                shift_reg[7:4] = shift_reg[7:4] + 4'd3;
            end
            // Ones digit (bits 3:0)
            if (shift_reg[3:0] >= 4'd5) begin
                shift_reg[3:0] = shift_reg[3:0] + 4'd3;
            end

   

In [58]:
sv_code = r"""
module binary_to_bcd_converter (
    input  [7:0]  binary_input,
    output [11:0] bcd_output
);

    reg [11:0] shift_reg;
    integer i;

    always @* begin
        shift_reg = 12'b0; // Initialize BCD digits to zero

        // Iterate through each bit of the binary_input, from MSB to LSB
        for (i = 0; i < 8; i = i + 1) begin
            // Apply +3 correction to any BCD digit that is 5 or greater
            // This is done before the shift operation for the current stage

            // Hundreds digit (bits 11:8)
            if (shift_reg[11:8] >= 4'd5) begin
                shift_reg[11:8] = shift_reg[11:8] + 4'd3;
            end
            // Tens digit (bits 7:4)
            if (shift_reg[7:4] >= 4'd5) begin
                shift_reg[7:4] = shift_reg[7:4] + 4'd3;
            end
            // Ones digit (bits 3:0)
            if (shift_reg[3:0] >= 4'd5) begin
                shift_reg[3:0] = shift_reg[3:0] + 4'd3;
            end

            // Shift the entire BCD register left by one bit
            // And insert the next bit from the binary_input (MSB first) into the LSB
            shift_reg = {shift_reg[10:0], binary_input[7-i]};
        end

        // After all iterations, shift_reg holds the final BCD result
        bcd_output = shift_reg;
    end

endmodule
"""
with open("binary_to_bcd.v", "w") as f:
    f.write(sv_code + "\n")
print("Wrote binary_to_bcd.v")

Wrote binary_to_bcd.v


In [59]:
TESTBENCH_URL = "https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v"

# Example format (do NOT use this unless it's your actual class link):
# TESTBENCH_URL = "https://raw.githubusercontent.com/<org>/<repo>/main/binary_to_bcd_tb.v"

if "PASTE_RAW_GITHUB_URL" in TESTBENCH_URL:
    raise RuntimeError("Set TESTBENCH_URL to the raw link for binary_to_bcd_tb.v from your course materials.")

!curl -L -o binary_to_bcd_tb.v "$TESTBENCH_URL"
!ls -lah

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0  10605      0 --:--:-- --:--:-- --:--:-- 10673
total 48K
drwxr-xr-x 1 root root 4.0K Mar  1 04:25 .
drwxr-xr-x 1 root root 4.0K Mar  1 01:05 ..
drwxr-xr-x 2 root root 4.0K Mar  1 01:41 abro_state_machine
-rw-r--r-- 1 root root 6.1K Mar  1 04:28 binary_to_bcd_claude_raw.txt
-rwxr-xr-x 1 root root 4.0K Mar  1 04:25 binary_to_bcd.out
-rw-r--r-- 1 root root 1.1K Mar  1 04:35 binary_to_bcd_tb.v
-rw-r--r-- 1 root root 1.3K Mar  1 04:35 binary_to_bcd.v
drwxr-xr-x 4 root root 4.0K Jan 16 14:24 .config
-rw-r--r-- 1 root root 3.2K Mar  1 04:25 my_design.vcd
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data
drwxr-xr-x 2 root root 4.0K Mar  1 01:38 shift_register


In [60]:
import subprocess

def sh(cmd: str):
    print(">>>", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout)
    if r.returncode != 0:
        if r.stderr.strip():
            print(r.stderr)
        raise RuntimeError(f"Command failed with exit code {r.returncode}")
    return r

sh("iverilog -g2012 -o binary_to_bcd.out binary_to_bcd.v binary_to_bcd_tb.v")
sh("vvp binary_to_bcd.out")

>>> iverilog -g2012 -o binary_to_bcd.out binary_to_bcd.v binary_to_bcd_tb.v
binary_to_bcd.v:37: error: bcd_output is not a valid l-value in tb_binary_to_bcd_converter.uut.
binary_to_bcd.v:4:      : bcd_output is declared here as wire.
binary_to_bcd_tb.v:8: warning: Port 1 (binary_input) of binary_to_bcd_converter expects 8 bits, got 5.
binary_to_bcd_tb.v:8:        : Padding 3 high bits of the port.
binary_to_bcd_tb.v:8: warning: Port 2 (bcd_output) of binary_to_bcd_converter expects 12 bits, got 8.
binary_to_bcd_tb.v:8:        : Padding 4 high bits of the port.
1 error(s) during elaboration.



RuntimeError: Command failed with exit code 1

In [ ]:
SPEC = """
Design a synthesizable Verilog/SystemVerilog module named binary_to_bcd_converter with:
- input  [7:0]  binary_input
- output [11:0] bcd_output

bcd_output format: {hundreds[3:0], tens[3:0], ones[3:0]}.
For binary_input 0..255, convert to decimal and output BCD digits.
Purely combinational (no clocks).
Use a synthesizable double-dabble (shift-add-3) implementation.
"""

PROMPT = f"""
You are an expert digital design engineer.

Task: Write synthesizable Verilog code that compiles with:
iverilog -g2012

CRITICAL OUTPUT RULES (you MUST follow these exactly):
1) Your ENTIRE response must be ONLY a single fenced code block: ```verilog ... ```
2) NO explanations, NO prose, NO comments outside the code block.
3) NO manual step-by-step solving. Output CODE only.
4) The code block must contain ONLY the module binary_to_bcd_converter — no testbench.
5) Purely combinational. No latches.
6) Use always @* with a compact for-loop (not unrolled stages).
7) Use a single shift_reg [11:0] and integer loop variable i.

CRITICAL IMPLEMENTATION RULES:
8) Use plain integers for comparisons and additions — NOT sized literals.
   CORRECT:   if (shift_reg[11:8] > 4)  shift_reg[11:8] = shift_reg[11:8] + 3;
   INCORRECT: if (shift_reg[11:8] >= 4'd5) shift_reg[11:8] = shift_reg[11:8] + 4'd3;
9) The add-3 check threshold MUST be > 4 (not >= 5) to avoid width inference bugs in iverilog.
10) The shift line MUST be: shift_reg = {{shift_reg[10:0], binary_input[7-i]}};
11) Loop runs exactly 8 times (i = 0 to 7), MSB first.
12) Initialize shift_reg = 12'b0 before the loop.

The EXACT loop body structure must be:
    if (shift_reg[11:8] > 4) shift_reg[11:8] = shift_reg[11:8] + 3;
    if (shift_reg[7:4]  > 4) shift_reg[7:4]  = shift_reg[7:4]  + 3;
    if (shift_reg[3:0]  > 4) shift_reg[3:0]  = shift_reg[3:0]  + 3;
    shift_reg = {{shift_reg[10:0], binary_input[7-i]}};

Spec:
{SPEC}
""".strip()

print(PROMPT[:800] + "\\n...\\n")

In [52]:
with open("binary_to_bcd_claude_raw.txt", "w") as f:
    f.write(completion.text)
print("Saved: binary_to_bcd_claude_raw.txt")

Saved: binary_to_bcd_claude_raw.txt


In [61]:
SPEC = """
Design a synthesizable Verilog/SystemVerilog module named binary_to_bcd_converter with:
- input  [7:0]  binary_input
- output [11:0] bcd_output

bcd_output format: {hundreds[3:0], tens[3:0], ones[3:0]}.
For binary_input 0..255, convert to decimal and output BCD digits.
Purely combinational (no clocks).
Use a synthesizable double-dabble (shift-add-3) implementation.
"""

PROMPT = f"""
You are an expert digital design engineer.

Task: Write synthesizable Verilog code that compiles with:
iverilog -g2012

CRITICAL OUTPUT RULES (you MUST follow these exactly):
1) Your ENTIRE response must be ONLY a single fenced code block: ```verilog ... ```
2) NO explanations, NO prose, NO comments outside the code block.
3) NO manual step-by-step solving. Output CODE only.
4) The code block must contain ONLY the module binary_to_bcd_converter — no testbench.
5) Purely combinational. No latches.
6) Use always @* with a compact for-loop (not unrolled stages).
7) Use a single shift_reg [11:0] and integer loop variable i.

CRITICAL IMPLEMENTATION RULES:
8) Use plain integers for comparisons and additions — NOT sized literals.
   CORRECT:   if (shift_reg[11:8] > 4)  shift_reg[11:8] = shift_reg[11:8] + 3;
   INCORRECT: if (shift_reg[11:8] >= 4'd5) shift_reg[11:8] = shift_reg[11:8] + 4'd3;
9) The add-3 check threshold MUST be > 4 (not >= 5) to avoid width inference bugs in iverilog.
10) The shift line MUST be: shift_reg = {{shift_reg[10:0], binary_input[7-i]}};
11) Loop runs exactly 8 times (i = 0 to 7), MSB first.
12) Initialize shift_reg = 12'b0 before the loop.

The EXACT loop body structure must be:
    if (shift_reg[11:8] > 4) shift_reg[11:8] = shift_reg[11:8] + 3;
    if (shift_reg[7:4]  > 4) shift_reg[7:4]  = shift_reg[7:4]  + 3;
    if (shift_reg[3:0]  > 4) shift_reg[3:0]  = shift_reg[3:0]  + 3;
    shift_reg = {{shift_reg[10:0], binary_input[7-i]}};

Spec:
{SPEC}
""".strip()

print(PROMPT[:800] + "\\n...\\n")

You are an expert digital design engineer.

Task: Write synthesizable Verilog code that compiles with:
iverilog -g2012

CRITICAL OUTPUT RULES (you MUST follow these exactly):
1) Your ENTIRE response must be ONLY a single fenced code block: ```verilog ... ```
2) NO explanations, NO prose, NO comments outside the code block.
3) NO manual step-by-step solving. Output CODE only.
4) The code block must contain ONLY the module binary_to_bcd_converter — no testbench.
5) Purely combinational. No latches.
6) Use always @* with a compact for-loop (not unrolled stages).
7) Use a single shift_reg [11:0] and integer loop variable i.

CRITICAL IMPLEMENTATION RULES:
8) Use plain integers for comparisons and additions — NOT sized literals.
   CORRECT:   if (shift_reg[11:8] > 4)  shift_reg[11:8] = shift_re\n...\n


In [62]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini model
model = genai.GenerativeModel('gemini-2.5-flash')
completion = model.generate_content(PROMPT)
print(completion.text)

```verilog
module binary_to_bcd_converter (
    input  [7:0]  binary_input,
    output [11:0] bcd_output
);

    reg [11:0] shift_reg;
    integer i;

    always @* begin
        shift_reg = 12'b0;

        for (i = 0; i < 8; i = i + 1) begin
            if (shift_reg[11:8] > 4) shift_reg[11:8] = shift_reg[11:8] + 3;
            if (shift_reg[7:4]  > 4) shift_reg[7:4]  = shift_reg[7:4]  + 3;
            if (shift_reg[3:0]  > 4) shift_reg[3:0]  = shift_reg[3:0]  + 3;
            shift_reg = {shift_reg[10:0], binary_input[7-i]};
        end
    end

    assign bcd_output = shift_reg;

endmodule
```


In [63]:
def extract_sv_code(text: str) -> str:
    # Prefer fenced systemverilog blocks
    m = re.search(r"```systemverilog\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if not m:
        # Fallback: any fenced block
        m = re.search(r"```(?:\w+)?\s*(.*?)```", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No fenced code block found in Claude output.")

    code = m.group(1).strip()

    # Sanity checks aligned with testbench expectations
    if "module binary_to_bcd_converter" not in code:
        raise ValueError("Extracted code does not contain 'module binary_to_bcd_converter'.")
    if "endmodule" not in code:
        raise ValueError("Extracted code does not contain 'endmodule'.")

    return code


sv_code = extract_sv_code(completion.text)
print("EXTRACTED SYSTEMVERILOG (preview)")
print(sv_code)

EXTRACTED SYSTEMVERILOG (preview)
module binary_to_bcd_converter (
    input  [7:0]  binary_input,
    output [11:0] bcd_output
);

    reg [11:0] shift_reg;
    integer i;

    always @* begin
        shift_reg = 12'b0;

        for (i = 0; i < 8; i = i + 1) begin
            if (shift_reg[11:8] > 4) shift_reg[11:8] = shift_reg[11:8] + 3;
            if (shift_reg[7:4]  > 4) shift_reg[7:4]  = shift_reg[7:4]  + 3;
            if (shift_reg[3:0]  > 4) shift_reg[3:0]  = shift_reg[3:0]  + 3;
            shift_reg = {shift_reg[10:0], binary_input[7-i]};
        end
    end

    assign bcd_output = shift_reg;

endmodule


In [64]:
sv_code = r"""
module binary_to_bcd_converter (
    input  [7:0]  binary_input,
    output [11:0] bcd_output
);

    reg [11:0] shift_reg;
    integer i;

    always @* begin
        shift_reg = 12'b0;

        for (i = 0; i < 8; i = i + 1) begin
            if (shift_reg[11:8] > 4) shift_reg[11:8] = shift_reg[11:8] + 3;
            if (shift_reg[7:4]  > 4) shift_reg[7:4]  = shift_reg[7:4]  + 3;
            if (shift_reg[3:0]  > 4) shift_reg[3:0]  = shift_reg[3:0]  + 3;
            shift_reg = {shift_reg[10:0], binary_input[7-i]};
        end
    end

    assign bcd_output = shift_reg;

endmodule
"""
with open("binary_to_bcd.v", "w") as f:
    f.write(sv_code + "\n")
print("Wrote binary_to_bcd.v")

Wrote binary_to_bcd.v


In [65]:
import subprocess

def sh(cmd: str):
    print(">>>", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout)
    if r.returncode != 0:
        if r.stderr.strip():
            print(r.stderr)
        raise RuntimeError(f"Command failed with exit code {r.returncode}")
    return r

sh("iverilog -g2012 -o binary_to_bcd.out binary_to_bcd.v binary_to_bcd_tb.v")
sh("vvp binary_to_bcd.out")

>>> iverilog -g2012 -o binary_to_bcd.out binary_to_bcd.v binary_to_bcd_tb.v
>>> vvp binary_to_bcd.out
Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!



CompletedProcess(args='vvp binary_to_bcd.out', returncode=0, stdout='Testing Binary-to-BCD Converter...\nVCD info: dumpfile my_design.vcd opened for output.\nAll test cases passed!\n', stderr='')